# Garbage Classification — ResNet18 Transfer Learning

## Kaggle Kurulumu
1. Sağ panelden **+ Add Data** butonuna tıklayın
2. Veri setinizi arayıp ekleyin (örn. `garbage-classification`)
3. Aşağıdaki `DATASET_NAME` değişkenine Kaggle'ın size atadığı **slug** adını girin
   - Örnek: dataset URL'si `.../datasets/kullanici/garbage-classification` ise slug = `garbage-classification`
4. Tüm hücreleri sırayla çalıştırın (**Run All**)

> Yerel ortamda da çalışır: kaynak klasör `dataset/raw/` olarak varsayılır.

In [ ]:
import os
import shutil
import random
from PIL import Image
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# ─── ORTAM & YOL AYARLARI ────────────────────────────────────────────────────
IS_KAGGLE = os.path.exists("/kaggle/input")

if IS_KAGGLE:
    source_dir = "/kaggle/input/datasets/asdasdasasdas/garbage-classification/Garbage classification/Garbage classification"
    base_dir   = "/kaggle/working/dataset"
else:
    source_dir = "../data/raw"
    base_dir   = "../data"

train_dir = os.path.join(base_dir, "train")
val_dir   = os.path.join(base_dir, "val")
test_dir  = os.path.join(base_dir, "test")

print(f"Ortam    : {'Kaggle' if IS_KAGGLE else 'Yerel'}")
print(f"Kaynak   : {source_dir}")
print(f"Train    : {train_dir}")
print(f"Val      : {val_dir}")
print(f"Test     : {test_dir}")

In [ ]:
classes = sorted([
    d for d in os.listdir(source_dir)
    if os.path.isdir(os.path.join(source_dir, d))
])
print("Bulunan sınıflar:", classes)

In [ ]:
for class_name in classes:
    class_path = os.path.join(source_dir, class_name)
    print(f"{class_name:12s} -> {len(os.listdir(class_path))} görüntü")

In [ ]:
# ─── VERİYİ BÖL (train / val / test) ─────────────────────────────────────────
# Hücre birden fazla çalıştırılırsa tekrar kopyalama yapılmaz
already_split = all(
    os.path.isdir(os.path.join(base_dir, s)) for s in ["train", "val", "test"]
)

if already_split:
    print("Veri zaten bölünmüş, bu adım atlanıyor.")
else:
    split_ratio = (0.70, 0.15, 0.15)
    random.seed(42)

    for class_name in classes:
        class_path = os.path.join(source_dir, class_name)
        if not os.path.isdir(class_path):
            continue

        images = os.listdir(class_path)
        random.shuffle(images)

        n       = len(images)
        n_train = int(n * split_ratio[0])
        n_val   = int(n * split_ratio[1])

        splits = {
            "train": images[:n_train],
            "val"  : images[n_train : n_train + n_val],
            "test" : images[n_train + n_val :],
        }

        for split_name, split_images in splits.items():
            target_dir = os.path.join(base_dir, split_name, class_name)
            os.makedirs(target_dir, exist_ok=True)
            for img_name in split_images:
                shutil.copy2(
                    os.path.join(class_path, img_name),
                    os.path.join(target_dir, img_name),
                )

    print("Veri ayırma tamamlandı.")

In [ ]:
for split in ["train", "val", "test"]:
    split_path = os.path.join(base_dir, split)
    print(f"\n{split.upper()} klasörü:")
    for class_name in sorted(os.listdir(split_path)):
        cp = os.path.join(split_path, class_name)
        if os.path.isdir(cp):
            print(f"  {class_name:12s} -> {len(os.listdir(cp))} görüntü")

In [ ]:
# Örnek görüntü göster
sample_path = None
for class_name in sorted(os.listdir(train_dir)):
    cp = os.path.join(train_dir, class_name)
    if os.path.isdir(cp):
        files = [f for f in os.listdir(cp) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
        if files:
            sample_path = os.path.join(cp, files[0])
            break

img = Image.open(sample_path)
print("Örnek dosya:", sample_path)
print("Boyut:", img.size, "| Mod:", img.mode)

plt.imshow(img)
plt.axis("off")
plt.show()

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Kullanılan cihaz:", device)

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

train_dataset = datasets.ImageFolder(root=train_dir, transform=transform)
val_dataset   = datasets.ImageFolder(root=val_dir,   transform=transform)
test_dataset  = datasets.ImageFolder(root=test_dir,  transform=transform)

# Windows'ta num_workers>0 notebook'ta sorun çıkarır; Kaggle (Linux) için 2 güvenlidir
_nw = 2 if IS_KAGGLE else 0
_pm = device.type == "cuda"  # pin_memory sadece CUDA'da faydalıdır

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=_nw, pin_memory=_pm)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=_nw, pin_memory=_pm)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=_nw, pin_memory=_pm)

print("Sınıf → indeks:", train_dataset.class_to_idx)

In [ ]:
images, labels = next(iter(train_loader))
print("Image shape:", images.shape)
print("Labels     :", labels[:10].tolist())

In [ ]:
# Tensörü görüntüye çevir (normalize tersine)
img_show = images[0].permute(1, 2, 0) * 0.5 + 0.5
img_show = img_show.clamp(0, 1)

plt.imshow(img_show)
plt.title(f"{train_dataset.classes[labels[0].item()]} (label={labels[0].item()})")
plt.axis("off")
plt.show()

## Basit CNN (Kıyaslama)

In [ ]:
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

cnn_model = SimpleCNN(num_classes=6).to(device)
print(cnn_model)

In [ ]:
import torch.optim as optim

criterion     = nn.CrossEntropyLoss()
cnn_optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)
epochs        = 3

for epoch in range(epochs):
    cnn_model.train()
    total_loss = 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        cnn_optimizer.zero_grad()
        loss = criterion(cnn_model(imgs), lbls)
        loss.backward()
        cnn_optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss / len(train_loader):.4f}")

In [ ]:
cnn_model.eval()
correct = total = 0
with torch.no_grad():
    for imgs, lbls in val_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        preds   = torch.argmax(cnn_model(imgs), dim=1)
        correct += (preds == lbls).sum().item()
        total   += lbls.size(0)
print(f"SimpleCNN Validation Accuracy: {correct / total:.4f}")

## ResNet18 — Transfer Learning

In [ ]:
from torchvision import models

# torchvision >= 0.13: weights= kullan (pretrained= deprecated)
try:
    from torchvision.models import ResNet18_Weights
    resnet = models.resnet18(weights=ResNet18_Weights.DEFAULT)
except ImportError:
    resnet = models.resnet18(pretrained=True)  # eski torchvision için fallback

# Son katmanı 6 sınıf için değiştir
resnet.fc = nn.Linear(resnet.fc.in_features, 6)
resnet    = resnet.to(device)

print("ResNet18 fc katmanı:", resnet.fc)

In [ ]:
resnet_optimizer = optim.Adam(resnet.parameters(), lr=0.0001)
epochs           = 3

for epoch in range(epochs):
    resnet.train()
    total_loss = 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        resnet_optimizer.zero_grad()
        loss = criterion(resnet(imgs), lbls)
        loss.backward()
        resnet_optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss / len(train_loader):.4f}")

In [ ]:
resnet.eval()
correct = total = 0
with torch.no_grad():
    for imgs, lbls in val_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        preds   = torch.argmax(resnet(imgs), dim=1)
        correct += (preds == lbls).sum().item()
        total   += lbls.size(0)
print(f"ResNet18 Validation Accuracy: {correct / total:.4f}")

In [ ]:
resnet.eval()
correct = total = 0
with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        preds   = torch.argmax(resnet(imgs), dim=1)
        correct += (preds == lbls).sum().item()
        total   += lbls.size(0)
print(f"ResNet18 Test Accuracy: {correct / total:.4f}")

---
## Sınıf Dengesizliği Analizi

Trash sınıfı train setinde en az görüntüye sahip sınıf olup paper gibi büyük sınıflara göre yaklaşık **4.3x** daha az temsil edilmektedir. Bu dengesizlik trash'in yanlış sınıflandırılmasına yol açabilir.

In [ ]:
import os, matplotlib.pyplot as plt

class_names  = sorted(os.listdir(train_dir))
class_counts = [len(os.listdir(os.path.join(train_dir, c))) for c in class_names]

colors = ['tomato' if c == 'trash' else 'steelblue' for c in class_names]
plt.figure(figsize=(10, 4))
bars = plt.bar(class_names, class_counts, color=colors)
for bar, cnt in zip(bars, class_counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             str(cnt), ha='center', fontsize=10)
plt.title("Train Setindeki Sınıf Dağılımı  (kırmızı = azınlık sınıfı)")
plt.ylabel("Görüntü Sayısı")
plt.tight_layout()
plt.show()

print(f"En fazla: {max(class_counts)}  |  En az: {min(class_counts)}  "
      f"|  Dengesizlik oranı: {max(class_counts)/min(class_counts):.1f}x")


## Dengesizlik Çözümü

3 katmanlı strateji:
- **Trash augmentation** — RandomCrop, Flip, Rotation, ColorJitter ile veri zenginleştirme
- **WeightedRandomSampler** — her batch'te sınıflar dengeli temsil edilir
- **Weighted CrossEntropyLoss** — nadir sınıflara kayıp hesabında daha fazla ağırlık

In [ ]:
from torchvision import transforms

base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

trash_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [ ]:
from torch.utils.data import Dataset
from torchvision.datasets import ImageFolder
from PIL import Image

class ClassAwareDataset(Dataset):
    def __init__(self, root, base_transform, trash_transform):
        self.dataset         = ImageFolder(root)
        self.base_transform  = base_transform
        self.trash_transform = trash_transform
        self.trash_idx       = self.dataset.class_to_idx['trash']
        self.targets         = self.dataset.targets

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        path, label = self.dataset.samples[idx]
        img         = Image.open(path).convert('RGB')
        t           = self.trash_transform if label == self.trash_idx else self.base_transform
        return t(img), label

print("ClassAwareDataset tanımlandı.")


In [ ]:
import torch, numpy as np
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets

balanced_train_dataset = ClassAwareDataset(train_dir, base_transform, trash_transform)

class_weights_arr = np.array([1.0 / c for c in class_counts])
sample_weights    = torch.DoubleTensor(
    [class_weights_arr[lbl] for lbl in balanced_train_dataset.targets]
)
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

_nw = 2 if IS_KAGGLE else 0
_pm = device.type == "cuda"

balanced_train_loader = DataLoader(balanced_train_dataset, batch_size=32,
                                   sampler=sampler, num_workers=_nw, pin_memory=_pm)
val_loader_b  = DataLoader(datasets.ImageFolder(val_dir,  transform=val_test_transform),
                           batch_size=32, num_workers=_nw, pin_memory=_pm)
test_loader_b = DataLoader(datasets.ImageFolder(test_dir, transform=val_test_transform),
                           batch_size=32, num_workers=_nw, pin_memory=_pm)

print("Dengeli loaderlar hazır.")
print({n: round(w, 5) for n, w in zip(class_names, class_weights_arr)})


## Dengeli Model Eğitimi — LR Scheduler · Early Stopping · Mixed Precision

| Teknik | Uygulama | Kazanım |
|--------|----------|---------|
| **CosineAnnealingLR** | LR, cosine eğrisiyle `1e-4 → 1e-6`'ya iner | Keskin minimumlardan kaçınır |
| **EarlyStopping** | Val loss `patience=5` epoch boyunca iyileşmezse durur | Overfitting önlenir |
| **Mixed Precision (AMP)** | FP16 ileri geçiş + FP32 ağırlık güncellemesi | ~1.5–2× GPU hızı |

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best       = None
        self.counter    = 0
        self.best_state = None

    def step(self, val_loss, model):
        if self.best is None or val_loss < self.best - self.min_delta:
            self.best       = val_loss
            self.counter    = 0
            self.best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            return False
        self.counter += 1
        return self.counter >= self.patience

    def restore(self, model):
        if self.best_state:
            model.load_state_dict(self.best_state)

print("EarlyStopping tanımlandı.")


In [ ]:
try:
    import wandb
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "wandb", "-q"], check=True)
    import wandb

PROJECT = "garbage-classification"
wandb.login()
print(f"W&B {wandb.__version__}  |  proje: '{PROJECT}'")


In [ ]:
from torchvision import models
import torch.nn as nn
import torch.optim as optim

use_amp = device.type == "cuda"

model_b     = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model_b.fc  = nn.Linear(model_b.fc.in_features, 6)
model_b     = model_b.to(device)

loss_w      = torch.FloatTensor(class_weights_arr / class_weights_arr.min()).to(device)
criterion_b = nn.CrossEntropyLoss(weight=loss_w)
optimizer_b = optim.Adam(model_b.parameters(), lr=1e-4)

MAX_EPOCHS = 30
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer_b, T_max=MAX_EPOCHS, eta_min=1e-6)
es         = EarlyStopping(patience=5)
scaler     = torch.cuda.amp.GradScaler(enabled=use_amp)

wandb.finish()
wandb.init(
    project=PROJECT, name="resnet18-balanced",
    config={"model": "resnet18", "max_epochs": MAX_EPOCHS, "lr": 1e-4,
            "batch_size": 32, "patience": 5, "scheduler": "CosineAnnealingLR",
            "amp": use_amp, "loss": "weighted_crossentropy",
            "sampler": "WeightedRandomSampler"}
)

for epoch in range(MAX_EPOCHS):
    model_b.train()
    train_loss = 0.0
    for imgs, lbls in balanced_train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer_b.zero_grad()
        with torch.autocast(device_type=device.type, enabled=use_amp):
            loss = criterion_b(model_b(imgs), lbls)
        scaler.scale(loss).backward()
        scaler.step(optimizer_b)
        scaler.update()
        train_loss += loss.item()

    model_b.eval()
    val_loss = correct = total = 0
    with torch.no_grad():
        for imgs, lbls in val_loader_b:
            imgs, lbls = imgs.to(device), lbls.to(device)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                out       = model_b(imgs)
                val_loss += criterion_b(out, lbls).item()
            correct += (out.argmax(1) == lbls).sum().item()
            total   += lbls.size(0)

    scheduler.step()
    avg_train = train_loss / len(balanced_train_loader)
    avg_val   = val_loss   / len(val_loader_b)
    val_acc   = correct / total
    lr_now    = scheduler.get_last_lr()[0]

    print(f"Epoch {epoch+1:2d}/{MAX_EPOCHS} | "
          f"Train: {avg_train:.4f} | Val: {avg_val:.4f} | "
          f"Acc: {val_acc:.4f} | LR: {lr_now:.2e}")

    wandb.log({"epoch": epoch+1, "train_loss": avg_train,
               "val_loss": avg_val, "val_acc": val_acc, "lr": lr_now})

    if es.step(avg_val, model_b):
        print(f"\nEarly stopping @ epoch {epoch+1}  (best val loss: {es.best:.4f})")
        es.restore(model_b)
        print("En iyi agırlıklar geri yüklendi.")
        break


## Confusion Matrix & Precision / Recall / F1

In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

model_b.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, lbls in test_loader_b:
        preds = model_b(imgs.to(device)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(lbls.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

cm = confusion_matrix(all_labels, all_preds)
fig_cm, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_title("Confusion Matrix — Test Seti")
ax.set_xlabel("Tahmin"); ax.set_ylabel("Gerçek")
plt.tight_layout(); plt.show()

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

test_acc = float((all_preds == all_labels).mean())
wandb.log({
    "test_accuracy":        test_acc,
    "confusion_matrix":     wandb.plot.confusion_matrix(
                                y_true=all_labels.tolist(),
                                preds=all_preds.tolist(),
                                class_names=class_names),
    "confusion_matrix_fig": wandb.Image(fig_cm),
})
print(f"W&B -> confusion matrix loglandı  (test acc: {test_acc:.4f})")


## Yanlış Sınıflandırılan Örnekler

Modelin hata yaptığı görüntüleri incelemek, hangi sınıf çiftlerinin birbirine karıştığını anlamak için en pratik yöntemdir.

In [ ]:
def denorm(t):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return (t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

model_b.eval()
wrong_imgs, wrong_true, wrong_pred = [], [], []
with torch.no_grad():
    for imgs, lbls in test_loader_b:
        preds = model_b(imgs.to(device)).argmax(1).cpu()
        mask  = preds != lbls
        wrong_imgs.extend(imgs[mask])
        wrong_true.extend(lbls[mask].numpy())
        wrong_pred.extend(preds[mask].numpy())

n = min(12, len(wrong_imgs))
print(f"Toplam hatalı tahmin: {len(wrong_imgs)} / {len(test_loader_b.dataset)}")

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for i, ax in enumerate(axes.flat):
    if i >= n:
        ax.axis('off'); continue
    ax.imshow(denorm(wrong_imgs[i]))
    ax.set_title(
        f"Gerçek:  {class_names[wrong_true[i]]}\nTahmin: {class_names[wrong_pred[i]]}",
        fontsize=8, color='darkred')
    ax.axis('off')

plt.suptitle("Yanlış Sınıflandırılan Örnekler", fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

wandb.log({"misclassified_count": len(wrong_imgs),
           "misclassified_examples": wandb.Image(fig)})


## Grad-CAM — Modelin Neye Baktığını Görselleştirme

`layer4[-1]` çıktısına hook takarak **sınıflandırma kararına en çok katkıda bulunan piksel bölgelerini** ısı haritası olarak gösteriyoruz. Kırmızı alan = modelin odaklandığı bölge.

In [ ]:
import numpy as np, matplotlib.cm as mpl_cm
from PIL import Image as PILImage

class GradCAM:
    def __init__(self, model, target_layer):
        self.model       = model
        self.activations = self.gradients = None
        self._fwd = target_layer.register_forward_hook(
            lambda m, i, o: setattr(self, 'activations', o.detach()))
        self._bwd = target_layer.register_full_backward_hook(
            lambda m, gi, go: setattr(self, 'gradients', go[0].detach()))

    def __call__(self, x, class_idx=None):
        out = self.model(x)
        idx = class_idx if class_idx is not None else out.argmax(1).item()
        self.model.zero_grad()
        out[0, idx].backward()
        w   = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam = torch.relu((w * self.activations).sum(1)).squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, idx

    def remove(self):
        self._fwd.remove(); self._bwd.remove()


def overlay_cam(img_tensor, cam):
    _mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    _std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img   = (img_tensor.cpu() * _std + _mean).clamp(0, 1).permute(1, 2, 0).numpy()
    cam_up = np.array(
        PILImage.fromarray((cam * 255).astype(np.uint8)).resize((224, 224), PILImage.BILINEAR)
    ) / 255.0
    return np.clip(0.55 * img + 0.45 * mpl_cm.jet(cam_up)[:, :, :3], 0, 1)


gradcam = GradCAM(model_b, model_b.layer4[-1])
model_b.eval()
samples = {i: None for i in range(6)}
for imgs, lbls in test_loader_b:
    for img, lbl in zip(imgs, lbls):
        l = lbl.item()
        if samples[l] is None:
            samples[l] = img
    if all(v is not None for v in samples.values()):
        break

_mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for cls_idx in range(6):
    img_t = samples[cls_idx]
    cam, pred_idx = gradcam(img_t.unsqueeze(0).to(device))
    orig  = (img_t * _std + _mean).clamp(0, 1).permute(1, 2, 0).numpy()

    axes[0, cls_idx].imshow(orig)
    axes[0, cls_idx].set_title(class_names[cls_idx], fontsize=10, fontweight='bold')
    axes[0, cls_idx].axis('off')

    color = 'green' if pred_idx == cls_idx else 'red'
    axes[1, cls_idx].imshow(overlay_cam(img_t, cam))
    axes[1, cls_idx].set_title(f"Pred: {class_names[pred_idx]}", fontsize=9, color=color)
    axes[1, cls_idx].axis('off')

axes[0, 0].set_ylabel("Orijinal", fontsize=11)
axes[1, 0].set_ylabel("Grad-CAM", fontsize=11)
plt.suptitle("Grad-CAM — Modelin Odaklandığı Bölgeler  (yeşil = doğru, kırmızı = hatalı)",
             fontsize=13)
plt.tight_layout(); plt.show()

wandb.log({"gradcam_heatmaps": wandb.Image(fig)})
wandb.finish()
print("resnet18-balanced W&B run tamamlandı.")
gradcam.remove()


---
## Model Benchmark — TIMM ile Karşılaştırma

Aynı `balanced_train_loader` + `weighted CrossEntropyLoss` ile 5 farklı mimari **15 epoch** (max) fine-tune edilir ve test setinde karşılaştırılır.

| Model | TIMM adı |
|---|---|
| ResNet-18 | `resnet18` |
| ResNet-50 | `resnet50` |
| EfficientNet-B0 | `efficientnet_b0` |
| MobileNetV3-Large | `mobilenetv3_large_100` |
| ViT-Tiny/16 | `vit_tiny_patch16_224` |

In [ ]:
try:
    import timm
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "timm", "-q"], check=True)
    import timm

import time

MODELS = [
    ("ResNet-18",       "resnet18"),
    ("ResNet-50",       "resnet50"),
    ("EfficientNet-B0", "efficientnet_b0"),
    ("MobileNetV3-L",   "mobilenetv3_large_100"),
    ("ViT-Tiny/16",     "vit_tiny_patch16_224"),
]


def benchmark_model(timm_name, display_name,
                    train_loader, val_loader, test_loader,
                    device, max_epochs=15, patience=4):
    print(f"\n── {display_name}  ({timm_name}) ──")
    use_amp  = device.type == "cuda"
    model    = timm.create_model(timm_name, pretrained=True, num_classes=6).to(device)
    params_m = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"   Parametre: {params_m:.2f}M")

    wandb.init(project=PROJECT, name=display_name,
               config={"model": timm_name, "params_M": round(params_m, 2),
                       "max_epochs": max_epochs, "patience": patience,
                       "lr": 1e-4, "scheduler": "CosineAnnealingLR", "amp": use_amp},
               reinit=True)

    criterion = torch.nn.CrossEntropyLoss(weight=loss_w)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    sched     = torch.optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=max_epochs, eta_min=1e-6)
    es_b      = EarlyStopping(patience=patience)
    scaler    = torch.cuda.amp.GradScaler(enabled=use_amp)

    for epoch in range(max_epochs):
        model.train()
        train_loss = 0.0
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            with torch.autocast(device_type=device.type, enabled=use_amp):
                loss = criterion(model(imgs), lbls)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            train_loss += loss.item()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                with torch.autocast(device_type=device.type, enabled=use_amp):
                    val_loss += criterion(model(imgs), lbls).item()

        sched.step()
        avg_val = val_loss   / len(val_loader)
        avg_tr  = train_loss / len(train_loader)
        lr_now  = sched.get_last_lr()[0]
        print(f"   Epoch {epoch+1:2d} | Train: {avg_tr:.4f} | Val: {avg_val:.4f} | LR: {lr_now:.2e}")
        wandb.log({"epoch": epoch+1, "train_loss": avg_tr, "val_loss": avg_val, "lr": lr_now})

        if es_b.step(avg_val, model):
            print(f"   Early stopping @ epoch {epoch+1}  (best val: {es_b.best:.4f})")
            es_b.restore(model); break

    model.eval()
    with torch.no_grad():
        dummy = torch.randn(1, 3, 224, 224).to(device)
        for _ in range(3): model(dummy)

    correct = total = 0
    times   = []
    with torch.no_grad():
        for imgs, lbls in test_loader:
            imgs = imgs.to(device)
            t0   = time.perf_counter()
            preds = model(imgs).argmax(1)
            if use_amp: torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1000 / imgs.size(0))
            correct += (preds.cpu() == lbls).sum().item()
            total   += lbls.size(0)

    acc = correct / total
    ms  = float(np.mean(times))
    print(f"   -> Acc: {acc:.4f}  |  {ms:.3f} ms/img  |  {params_m:.2f}M params")
    wandb.log({"test_accuracy": acc, "inference_ms_per_img": ms, "params_M": params_m})
    wandb.finish()
    return {"Model": display_name, "Params (M)": round(params_m, 2),
            "ms/img": round(ms, 3), "Acc %": round(acc * 100, 2)}

print(f"TIMM {timm.__version__}  |  {len(MODELS)} model hazır.")


In [ ]:
results = []
for display_name, timm_name in MODELS:
    r = benchmark_model(timm_name, display_name,
                        balanced_train_loader, val_loader_b, test_loader_b, device)
    results.append(r)
print("\nBenchmark tamamlandı.")


In [ ]:
import pandas as pd

df = pd.DataFrame(results).set_index("Model")

print("\n" + "=" * 65)
print(f"  {'Model':<22} {'Params (M)':>10} {'ms/img':>10} {'Acc %':>10}")
print("=" * 65)
for name, row in df.iterrows():
    print(f"  {name:<22} {row['Params (M)']:>10.2f} {row['ms/img']:>10.3f} {row['Acc %']:>9.2f}%")
print("=" * 65)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
x    = range(len(df))
lbls = df.index.tolist()
specs = [("Params (M)", "steelblue",  "Milyon Parametre"),
         ("ms/img",     "darkorange", "ms / görüntü"),
         ("Acc %",      "seagreen",   "Doğruluk (%)")]

for ax, (col, color, ylabel) in zip(axes, specs):
    bars = ax.bar(x, df[col], color=color)
    ax.set_xticks(x); ax.set_xticklabels(lbls, rotation=22, ha='right', fontsize=9)
    ax.set_ylabel(ylabel); ax.set_title(col)
    for bar, v in zip(bars, df[col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.015,
                f"{v:.1f}", ha='center', va='bottom', fontsize=8)

plt.suptitle("Model Benchmark Karşılaştırması", fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()
